# Querying the Hugging Face Hub with a Tiny LLM and FastMCP

Load Qwen2.5-0.5B-Instruct, read live data from a FastMCP server, and answer questions about the Hugging Face Hub catalogue via context injection.

[Read the full article](https://jheiduk.com/posts/tiny-llm-mcp-hub-client/)

In [1]:
# MCP server + Hugging Face Hub client
!pip install fastmcp huggingface_hub

# LLM inference
!pip install transformers torch

# Required for asyncio.run() inside Jupyter's existing event loop
!pip install nest_asyncio


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Write the FastMCP Server

The server exposes four MCP resources wrapping the Hugging Face Hub API. Writing it to disk here makes the notebook self-contained — `Client("hf_server.py")` will launch it as a subprocess.

In [2]:
server_code = '''
from fastmcp import FastMCP
from huggingface_hub import HfApi
import json

mcp = FastMCP("HuggingFace Hub Explorer")
api = HfApi()


@mcp.resource("hf://models")
def list_models() -> str:
    """Top 10 most liked text-generation models on Hugging Face Hub."""
    models = list(api.list_models(filter="text-generation", sort="likes", direction=-1, limit=10))
    return json.dumps([
        {"id": m.id, "likes": m.likes or 0, "downloads": m.downloads or 0}
        for m in models
    ], indent=2)


@mcp.resource("hf://datasets")
def list_datasets() -> str:
    """Top 10 most liked datasets on Hugging Face Hub."""
    datasets = list(api.list_datasets(sort="likes", direction=-1, limit=10))
    return json.dumps([
        {"id": d.id, "likes": d.likes or 0, "downloads": d.downloads or 0}
        for d in datasets
    ], indent=2)


@mcp.resource("hf://models/{owner}/{name}")
def get_model(owner: str, name: str) -> str:
    """Fetch metadata for a specific model from Hugging Face Hub."""
    info = api.model_info(f"{owner}/{name}")
    return json.dumps({
        "id": info.id,
        "author": info.author,
        "task": info.pipeline_tag or "unknown",
        "likes": info.likes,
        "downloads": info.downloads,
        "tags": info.tags[:15],
    }, indent=2)


@mcp.resource("hf://datasets/{owner}/{name}")
def get_dataset(owner: str, name: str) -> str:
    """Fetch metadata for a specific dataset from Hugging Face Hub."""
    info = api.dataset_info(f"{owner}/{name}")
    return json.dumps({
        "id": info.id,
        "author": info.author,
        "likes": info.likes,
        "downloads": info.downloads,
        "tags": info.tags[:15],
    }, indent=2)


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("hf_server.py", "w", encoding="utf-8") as f:
    f.write(server_code.strip())

print("Server written to hf_server.py")


Server written to hf_server.py


## Load the Model

Qwen2.5-0.5B-Instruct is instruction-tuned and ships with a chat template. It downloads ~1 GB on first use and runs without a GPU.

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,   # use torch.bfloat16 if you have a GPU
)
model.eval()
print("Model loaded.")

c:\Users\julie\jheiduk.com\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 492.39it/s, Materializing param=model.norm.weight]                              


Model loaded.


## Read a Resource from the MCP Server

`PythonStdioTransport` spawns `hf_server.py` as a subprocess with an explicit `log_file`, which avoids the `fileno` `UnsupportedOperation` error that occurs when Jupyter's `ipykernel` `OutStream` is used as stderr. `nest_asyncio` lets `asyncio.run()` work inside Jupyter's existing event loop.


In [4]:
import asyncio
import sys
import nest_asyncio
from fastmcp import Client
from fastmcp.client.transports import PythonStdioTransport

nest_asyncio.apply()


async def fetch_resource(uri: str) -> str:
    """Fetch a resource from the MCP server.

    Uses PythonStdioTransport with an explicit log_file to avoid the
    `fileno` UnsupportedOperation error that occurs when Jupyter's
    ipykernel OutStream is passed as stderr to subprocess.Popen.
    """
    transport = PythonStdioTransport(
        script_path="hf_server.py",
        python_cmd=sys.executable,
        log_file=open("mcp_server.log", "w"),  # real file → has fileno()
    )
    async with Client(transport) as client:
        result = await client.read_resource(uri)
    return result[0].text


# Quick smoke-test
catalogue = asyncio.run(fetch_resource("hf://models"))
print(catalogue[:300])


[
  {
    "id": "deepseek-ai/DeepSeek-R1",
    "likes": 13021,
    "downloads": 733701
  },
  {
    "id": "meta-llama/Meta-Llama-3-8B",
    "likes": 6466,
    "downloads": 1812133
  },
  {
    "id": "meta-llama/Llama-3.1-8B-Instruct",
    "likes": 5491,
    "downloads": 6157607
  },
  {
    "id": "b


## Inject Context and Generate an Answer

The resource JSON goes into the system message; the user's question goes in the user turn. `apply_chat_template` formats the messages with Qwen2.5's special tokens.

In [5]:
def ask(question: str, context: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant with access to a Hugging Face Hub catalogue.\n\n"
                f"Catalogue (JSON):\n{context}"
            ),
        },
        {"role": "user", "content": question},
    ]

    # apply_chat_template formats messages with Qwen2.5's special tokens
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,               # greedy for reproducibility
            pad_token_id=tokenizer.eos_token_id,
        )

    # slice off the echoed prompt tokens before decoding
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

## Full Pipeline

Fetch two resources from the MCP server and ask the LLM a question about each.

In [6]:
async def main():
    # 1. Top models catalogue → find the most downloaded
    catalogue = await fetch_resource("hf://models")
    answer = ask(
        "Which model in the catalogue has the most downloads? Give the model ID and the download count.",
        catalogue,
    )
    print("Most downloaded model:")
    print(answer)

    # 2. Specific model detail → extract task and tags
    detail = await fetch_resource("hf://models/google/gemma-2-2b")
    answer2 = ask(
        "What task is this model designed for? List three of its tags.",
        detail,
    )
    print("\nGemma-2-2b detail:")
    print(answer2)


asyncio.run(main())


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Most downloaded model:
The model with the highest number of downloads is "meta-llama/Llama-2-7b-chat-hf", which has 317,632 downloads.

Gemma-2-2b detail:
The task is designed for text generation. Three of its tags are transformers, safetensors, and gemma2.
